In [1]:
import mediapipe as mp
import numpy as np
import cv2

print("mediapipe:", mp.__version__)
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)

mediapipe: 0.10.32
numpy: 2.4.1
opencv: 4.11.0


# Understand & Isolate Core Components

## Phase 1

File map from TGCN:
- **tgcn_model.py**: The core model definition. This is what we will ultimately extend.
- **layers.py**: Graph convolution primitives. Reusable, probably won't be unchanged.
- **models.py**: Wrappers / model variants. Some parts reusable, some legacy.
- **utils.py**: Label encoding, helpers. Mostly reusable.
- **sign_dataset.py**: Dataset + sampling logic. Needs major simplification for our TRM pipeline.
- **gen_features.py**: Feature generation / caching logic. Inspiration only — needs to be streamlined.
- **videotransforms.py**: Temporal augmentations. Optional for now; might be useful later.

In [42]:
import os
import sys
from pathlib import Path
import numpy as np

import pandas as pd

import torch

PROJECT_ROOT = Path(".").resolve()
TGCN_CODE_DIR = f"{PROJECT_ROOT}/code/TGCN"
CSV_PATH = PROJECT_ROOT / "extracted/asl_hand_landmarks_per_frame.csv"

sys.path.append(str(TGCN_CODE_DIR))

from tgcn_model import GCN_muti_att

In [44]:
def get_device(prefer_mps=True):
    if prefer_mps and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device(prefer_mps=True)
print("Using device:", device)

Using device: mps


In [46]:
BODY_JOINTS = 13
HAND_JOINTS = 21
TOTAL_JOINTS = BODY_JOINTS + 2 * HAND_JOINTS  # 55

def _is_normalized_0_1(values: np.ndarray) -> bool:
    """Heuristic: MediaPipe often outputs normalized [0,1]."""
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return False
    return finite.max() <= 1.5  # generous

def _to_256_space_xy(xy: np.ndarray, assume_normalized: bool) -> np.ndarray:
    """
    xy: (..., 2) in either [0,1] or pixels
    returns pixels in a 256x256 coordinate space
    """
    if assume_normalized:
        return xy * 256.0
    return xy

def _normalize_to_minus1_plus1(xy_256: np.ndarray) -> np.ndarray:
    """
    Maps pixel coords in [0..256] to [-1..+1] used by Pose-GCN/TGCN preprocessing.
    """
    return 2.0 * ((xy_256 / 256.0) - 0.5)

def row_to_pose55_xy(row: pd.Series) -> np.ndarray:
    """
    Returns pose for ONE frame: shape (55, 2) with (x,y) normalized to [-1,1]
    Body joints are zeros (13x2).
    """
    pose = np.zeros((TOTAL_JOINTS, 2), dtype=np.float32)

    # Determine if CSV uses normalized coords (0..1) for hands
    # (check a few left/right x columns if present)
    sample_cols = []
    for side in ("left", "right"):
        col = f"{side}_lm0_x"
        if col in row.index:
            sample_cols.append(col)
    sample_vals = np.array([row[c] for c in sample_cols], dtype=np.float32) if sample_cols else np.array([2.0])
    assume_normalized = _is_normalized_0_1(sample_vals)

    # ---- Left hand (joints 13..33) ----
    left_present = int(row.get("left_present", 0))
    if left_present == 1:
        for j in range(HAND_JOINTS):
            x = row.get(f"left_lm{j}_x", np.nan)
            y = row.get(f"left_lm{j}_y", np.nan)
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            xy = np.array([x, y], dtype=np.float32)
            xy_256 = _to_256_space_xy(xy, assume_normalized=assume_normalized)
            xy_norm = _normalize_to_minus1_plus1(xy_256)
            pose[BODY_JOINTS + j] = xy_norm

    # ---- Right hand (joints 34..54) ----
    right_present = int(row.get("right_present", 0))
    if right_present == 1:
        for j in range(HAND_JOINTS):
            x = row.get(f"right_lm{j}_x", np.nan)
            y = row.get(f"right_lm{j}_y", np.nan)
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            xy = np.array([x, y], dtype=np.float32)
            xy_256 = _to_256_space_xy(xy, assume_normalized=assume_normalized)
            xy_norm = _normalize_to_minus1_plus1(xy_256)
            pose[BODY_JOINTS + HAND_JOINTS + j] = xy_norm

    return pose

In [48]:
def sample_frame_indices(num_frames: int, T: int, strategy="uniform"):
    """
    Returns list of frame indices of length T (with padding if needed).
    """
    if num_frames <= 0:
        return [0] * T

    if strategy == "uniform":
        if num_frames >= T:
            # evenly spaced
            idx = np.linspace(0, num_frames - 1, T).round().astype(int).tolist()
        else:
            # pad last
            idx = list(range(num_frames)) + [num_frames - 1] * (T - num_frames)
        return idx

    if strategy == "first":
        idx = list(range(min(T, num_frames)))
        if len(idx) < T:
            idx += [num_frames - 1] * (T - len(idx))
        return idx

    raise ValueError(f"Unknown strategy: {strategy}")

def csv_to_tgcn_input(csv_path: Path, T=32, sampling="uniform"):
    """
    Returns:
      X: torch.FloatTensor shape (1, 55, 2*T)
      meta: dict with frames used
    """
    df = pd.read_csv(csv_path)
    if df.shape[0] == 0:
        raise ValueError(f"CSV has 0 rows: {csv_path}")

    idxs = sample_frame_indices(len(df), T, strategy=sampling)

    # Build (55, 2*T)
    seq = np.zeros((TOTAL_JOINTS, 2 * T), dtype=np.float32)

    for t, row_idx in enumerate(idxs):
        pose55 = row_to_pose55_xy(df.iloc[row_idx])  # (55,2)
        seq[:, 2*t:2*t+2] = pose55

    X = torch.from_numpy(seq).unsqueeze(0)  # (1,55,2T)
    meta = {
        "num_frames_csv": len(df),
        "T": T,
        "frame_indices": idxs,
    }
    return X, meta

In [50]:
T = 32

X, meta = csv_to_tgcn_input(CSV_PATH, T=T, sampling="uniform")
print("Input shape:", tuple(X.shape))  # expect (1, 55, 64) if T=32

# ---- model hyperparams ----
# input_feature MUST equal 2*T because we concatenated (x,y) across T frames
input_feature = 2 * T

hidden_feature = 256     # you can tune later
num_classes = 2000       # set to your class count (WLASL2000 = 2000)
p_dropout = 0.3
num_stage = 2            # number of GC blocks

model = GCN_muti_att(
    input_feature=input_feature,
    hidden_feature=hidden_feature,
    num_class=num_classes,
    p_dropout=p_dropout,
    num_stage=num_stage,
    is_resi=True
).to(device)

model.eval()

with torch.no_grad():
    logits = model(X.to(device))  # (1, num_classes)
    probs = torch.softmax(logits, dim=-1)
    topk = torch.topk(probs, k=10, dim=-1)

print("Logits shape:", tuple(logits.shape))
print("Top-10 class ids:", topk.indices[0].cpu().tolist())
print("Top-10 probs:", [float(x) for x in topk.values[0].cpu().tolist()])
print("Meta:", meta)

Input shape: (1, 55, 64)
Logits shape: (1, 2000)
Top-10 class ids: [375, 912, 207, 1009, 148, 1624, 137, 514, 1016, 1391]
Top-10 probs: [0.0005915982183068991, 0.0005754762678407133, 0.0005748845869675279, 0.0005748268449679017, 0.0005729145486839116, 0.0005718700704164803, 0.0005690049147233367, 0.0005669582169502974, 0.0005661676404997706, 0.0005656973808072507]
Meta: {'num_frames_csv': 105, 'T': 32, 'frame_indices': [0, 3, 7, 10, 13, 17, 20, 23, 27, 30, 34, 37, 40, 44, 47, 50, 54, 57, 60, 64, 67, 70, 74, 77, 81, 84, 87, 91, 94, 97, 101, 104]}


In [52]:
from dataclasses import dataclass
from typing import Dict, Any, Optional, List

@dataclass
class Pred:
    csv: str
    frames_csv: int
    T: int
    topk_ids: List[int]
    topk_probs: List[float]
    frame_indices: List[int]

def predict_one_csv(csv_path, T=32, topk=10, sampling="uniform") -> Pred:
    X, meta = csv_to_tgcn_input(Path(csv_path), T=T, sampling=sampling)
    model.eval()
    with torch.no_grad():
        logits = model(X.to(device))
        probs = torch.softmax(logits, dim=-1)
        tk = torch.topk(probs, k=topk, dim=-1)

    ids = tk.indices[0].detach().cpu().numpy().tolist()
    ps  = tk.values[0].detach().cpu().numpy().tolist()

    return Pred(
        csv=str(csv_path),
        frames_csv=meta["num_frames_csv"],
        T=meta["T"],
        topk_ids=ids,
        topk_probs=[float(x) for x in ps],
        frame_indices=meta["frame_indices"],
    )

In [60]:
from pathlib import Path
import pandas as pd

CSV_DIR = Path("extracted")
T = 32
TOPK = 10
MAX_FILES = 5

csv_files = sorted(CSV_DIR.glob("*.csv"))
print("Found CSVs:", len(csv_files))
for p in csv_files[:10]:
    print(" -", p.name)

results = []
for i, csv_path in enumerate(csv_files[:MAX_FILES], start=1):
    pred = predict_one_csv(csv_path, T=T, topk=TOPK, sampling="uniform")

    print(f"\n[{i}/{min(MAX_FILES, len(csv_files))}] {Path(pred.csv).name}")
    print("  frames:", pred.frames_csv, "T:", pred.T)
    print("  top-10 ids :", pred.topk_ids)
    print("  top-10 prob:", [round(x, 6) for x in pred.topk_probs])

    results.append({
        "file": Path(pred.csv).name,
        "frames": pred.frames_csv,
        "T": pred.T,
        "top1_id": pred.topk_ids[0],
        "top1_prob": pred.topk_probs[0],
        "top5_ids": pred.topk_ids[:5],
        "top5_probs": pred.topk_probs[:5],
    })

df_preds = pd.DataFrame(results)
df_preds

Found CSVs: 1
 - asl_hand_landmarks_per_frame.csv

[1/1] asl_hand_landmarks_per_frame.csv
  frames: 105 T: 32
  top-10 ids : [375, 912, 207, 1009, 148, 1624, 137, 514, 1016, 1391]
  top-10 prob: [0.000592, 0.000575, 0.000575, 0.000575, 0.000573, 0.000572, 0.000569, 0.000567, 0.000566, 0.000566]


,file,frames,T,top1_id,top1_prob,top5_ids,top5_probs
0,asl_hand_landmarks_per_frame.csv,105,32,375,0.000592,"[375, 912, 207, 1009, 148]","[0.0005915982183068991, 0.0005754762678407133,..."


In [62]:
SINGLE_CSV = Path("extracted/asl_hand_landmarks_per_frame.csv")
T = 32

for sampling in ["uniform", "first"]:
    pred = predict_one_csv(SINGLE_CSV, T=T, topk=10, sampling=sampling)
    print(f"\nSampling={sampling} | frames={pred.frames_csv} | T={pred.T}")
    print("Top-10:", pred.topk_ids)
    print("Prob  :", [round(x, 6) for x in pred.topk_probs])


Sampling=uniform | frames=105 | T=32
Top-10: [375, 912, 207, 1009, 148, 1624, 137, 514, 1016, 1391]
Prob  : [0.000592, 0.000575, 0.000575, 0.000575, 0.000573, 0.000572, 0.000569, 0.000567, 0.000566, 0.000566]

Sampling=first | frames=105 | T=32
Top-10: [375, 148, 1009, 1624, 912, 207, 514, 137, 1016, 1391]
Prob  : [0.00059, 0.000575, 0.000574, 0.000573, 0.000573, 0.000573, 0.000568, 0.000567, 0.000566, 0.000565]


In [66]:
from pathlib import Path
import pandas as pd

SINGLE_CSV = Path("extracted/asl_hand_landmarks_per_frame.csv")
OUT_PATH = Path("output/tgcn_preds_sampling_compare.csv")

rows = []
for sampling in ["uniform", "first"]:
    pred = predict_one_csv(SINGLE_CSV, T=32, topk=10, sampling=sampling)
    rows.append({
        "file": Path(pred.csv).name,
        "sampling": sampling,
        "frames": pred.frames_csv,
        "T": pred.T,
        "top10_ids": str(pred.topk_ids),
        "top10_probs": str([float(x) for x in pred.topk_probs]),
        "frame_indices": str(pred.frame_indices),
        "top1_id": pred.topk_ids[0],
        "top1_prob": float(pred.topk_probs[0]),
    })

df_compare = pd.DataFrame(rows)
df_compare.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH.resolve())
print(df_compare)

Saved: /Users/dru/Downloads/Dumper/MSUAZ/capstone/trials/output/tgcn_preds_sampling_compare.csv
                               file sampling  frames   T  \
0  asl_hand_landmarks_per_frame.csv  uniform     105  32   
1  asl_hand_landmarks_per_frame.csv    first     105  32   

                                           top10_ids  \
0  [375, 912, 207, 1009, 148, 1624, 137, 514, 101...   
1  [375, 148, 1009, 1624, 912, 207, 514, 137, 101...   

                                         top10_probs  \
0  [0.0005915982183068991, 0.0005754762678407133,...   
1  [0.0005903896526433527, 0.000574620149563998, ...   

                                       frame_indices  top1_id  top1_prob  
0  [0, 3, 7, 10, 13, 17, 20, 23, 27, 30, 34, 37, ...      375   0.000592  
1  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...      375   0.000590  


## Real thing starts here

In [71]:
from pathlib import Path
import os
import csv
import cv2
import numpy as np
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ---- Paths (edit if needed) ----
VIDEOS_DIR = Path("videos")
OUT_DIR = Path("output")
MODEL_PATH = Path("models/hand_landmarker.task")

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Videos dir:", VIDEOS_DIR.resolve())
print("Out dir   :", OUT_DIR.resolve())
print("Model     :", MODEL_PATH.resolve())

Videos dir: /Users/dru/Downloads/Dumper/MSUAZ/capstone/trials/videos
Out dir   : /Users/dru/Downloads/Dumper/MSUAZ/capstone/trials/output
Model     : /Users/dru/Downloads/Dumper/MSUAZ/capstone/trials/models/hand_landmarker.task


In [73]:
BaseOptions = python.BaseOptions
HandLandmarkerOptions = vision.HandLandmarkerOptions
HandLandmarker = vision.HandLandmarker
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

print("HandLandmarker options ready.")

HandLandmarker options ready.


In [75]:
HAND_JOINTS = 21

def make_header():
    cols = ["frame", "timestamp_ms", "left_present", "right_present", "left_score", "right_score"]
    for side in ("left", "right"):
        for i in range(HAND_JOINTS):
            cols += [f"{side}_lm{i}_x", f"{side}_lm{i}_y", f"{side}_lm{i}_z"]
    return cols

CSV_HEADER = make_header()
len(CSV_HEADER), CSV_HEADER[:12], CSV_HEADER[-6:]

(132,
 ['frame',
  'timestamp_ms',
  'left_present',
  'right_present',
  'left_score',
  'right_score',
  'left_lm0_x',
  'left_lm0_y',
  'left_lm0_z',
  'left_lm1_x',
  'left_lm1_y',
  'left_lm1_z'],
 ['right_lm19_x',
  'right_lm19_y',
  'right_lm19_z',
  'right_lm20_x',
  'right_lm20_y',
  'right_lm20_z'])

In [77]:
def empty_hand_row():
    """Returns 21*3 NaNs for a single hand."""
    return [np.nan] * (HAND_JOINTS * 3)

def flatten_landmarks(hand_lms):
    """hand_lms: list of 21 landmarks -> [x0,y0,z0,x1,y1,z1,...]"""
    out = []
    for lm in hand_lms:
        out.extend([float(lm.x), float(lm.y), float(lm.z)])
    return out

def parse_result_to_lr(result):
    """
    Returns:
      left_present (0/1), right_present (0/1),
      left_score (float), right_score (float),
      left_vals (63), right_vals (63)
    """
    left_present = 0
    right_present = 0
    left_score = 0.0
    right_score = 0.0
    left_vals = empty_hand_row()
    right_vals = empty_hand_row()

    if not result.hand_landmarks:
        return left_present, right_present, left_score, right_score, left_vals, right_vals

    # MediaPipe tasks typically also returns handedness for each detected hand
    handedness_list = getattr(result, "handedness", None)

    for idx, hand_lms in enumerate(result.hand_landmarks):
        label = None
        score = 0.0

        if handedness_list and len(handedness_list) > idx and handedness_list[idx]:
            # handedness_list[idx] is a list of classifications; take best
            cls = handedness_list[idx][0]
            label = cls.category_name  # "Left" or "Right"
            score = float(cls.score)

        vals = flatten_landmarks(hand_lms)

        if label == "Left":
            left_present, left_score, left_vals = 1, score, vals
        elif label == "Right":
            right_present, right_score, right_vals = 1, score, vals
        else:
            # Fallback if handedness missing: fill first empty slot
            if left_present == 0:
                left_present, left_score, left_vals = 1, score, vals
            elif right_present == 0:
                right_present, right_score, right_vals = 1, score, vals

    return left_present, right_present, left_score, right_score, left_vals, right_vals

In [93]:
def process_video_to_csv(video_path: Path, out_csv: Path, landmarker, max_frames=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("❌ Could not open:", video_path)
        return {"video": video_path.name, "status": "failed_open", "rows": 0}

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30.0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    rows_written = 0
    bad_frames = 0

    with out_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)

        frame_idx = 0
        while True:
            if max_frames is not None and frame_idx >= max_frames:
                break

            ok, frame_bgr = cap.read()
            if not ok:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            timestamp_ms = int((frame_idx / fps) * 1000)

            try:
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                result = landmarker.detect_for_video(mp_image, timestamp_ms)
            except Exception as e:
                bad_frames += 1
                frame_idx += 1
                continue

            left_present, right_present, left_score, right_score, left_vals, right_vals = parse_result_to_lr(result)

            row = [
                frame_idx,
                timestamp_ms,
                left_present,
                right_present,
                left_score,
                right_score,
                *left_vals,
                *right_vals,
            ]
            writer.writerow(row)
            rows_written += 1
            frame_idx += 1

    cap.release()

    return {
        "video": video_path.name,
        "status": "ok",
        "rows": rows_written,
        "bad_frames": bad_frames,
        "fps": fps,
        "total_frames": total_frames,
        "out_csv": str(out_csv),
    }

In [95]:
import csv
import cv2
import numpy as np
from pathlib import Path
import mediapipe as mp

def process_video_to_csv_safe(video_path: Path, out_csv: Path, landmarker, max_frames=None):
    """
    Writes per-frame landmarks into out_csv.
    Uses a temp file and renames only if at least 1 data row is written.
    Tries AVFoundation first on macOS, falls back to default backend.
    """

    # Prefer AVFoundation on macOS (works even if OpenCV default backend is flaky)
    cap = cv2.VideoCapture(str(video_path), cv2.CAP_AVFOUNDATION)
    if not cap.isOpened():
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            return {"video": video_path.name, "status": "failed_open", "rows": 0, "out_csv": str(out_csv)}

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 30.0

    tmp_csv = out_csv.with_suffix(".csv.tmp")

    rows_written = 0
    bad_frames = 0

    with tmp_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)

        frame_idx = 0

        # Try decoding first frame immediately: if it fails, don't leave header-only files behind
        ok, frame_bgr = cap.read()
        if not ok or frame_bgr is None:
            cap.release()
            tmp_csv.unlink(missing_ok=True)
            return {
                "video": video_path.name,
                "status": "failed_decode_first_frame",
                "rows": 0,
                "out_csv": str(out_csv),
            }

        while True:
            if max_frames is not None and frame_idx >= max_frames:
                break

            if frame_idx > 0:
                ok, frame_bgr = cap.read()
                if not ok or frame_bgr is None:
                    break

            try:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                timestamp_ms = int((frame_idx / fps) * 1000)

                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                result = landmarker.detect_for_video(mp_image, timestamp_ms)

                left_present, right_present, left_score, right_score, left_vals, right_vals = parse_result_to_lr(result)

                row = [
                    frame_idx,
                    timestamp_ms,
                    left_present,
                    right_present,
                    left_score,
                    right_score,
                    *left_vals,
                    *right_vals,
                ]
                writer.writerow(row)
                rows_written += 1

            except Exception:
                bad_frames += 1

            frame_idx += 1

    cap.release()

    # If no data rows, delete temp and report
    if rows_written == 0:
        tmp_csv.unlink(missing_ok=True)
        return {"video": video_path.name, "status": "no_rows", "rows": 0, "out_csv": str(out_csv)}

    # Atomic rename temp -> final
    tmp_csv.replace(out_csv)

    return {
        "video": video_path.name,
        "status": "ok",
        "rows": rows_written,
        "bad_frames": bad_frames,
        "fps": fps,
        "out_csv": str(out_csv),
    }

In [107]:
import os
import cv2
import csv
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import mediapipe as mp

VIDEOS_DIR = Path("videos")
OUT_DIR = Path("output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_VIDEOS = 20
video_files = sorted(VIDEOS_DIR.glob("*.mp4"))[:MAX_VIDEOS]

# ---------- helpers ----------

def csv_has_data_rows(csv_path: Path) -> bool:
    try:
        with csv_path.open("r", encoding="utf-8") as f:
            if next(f, None) is None:
                return False
            return next(f, None) is not None
    except Exception:
        return False


def to63(lms21):
    arr = np.array([[lm.x, lm.y, lm.z] for lm in lms21], dtype=np.float32)
    return arr.reshape(-1)


CSV_HEADER = (
    ["frame", "timestamp_ms",
     "left_present", "right_present",
     "left_score", "right_score"] +
    [f"left_lm{i}_{a}" for i in range(21) for a in ("x", "y", "z")] +
    [f"right_lm{i}_{a}" for i in range(21) for a in ("x", "y", "z")]
)

nan63 = np.full((63,), np.nan, dtype=np.float32)


def process_video_to_csv_working(video_path: Path, out_csv: Path, landmarker):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"video": video_path.name, "status": "failed_open", "rows": 0}

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    tmp_csv = out_csv.with_suffix(".csv.tmp")
    rows_written = 0
    last_ts = -1  # <--- ADD THIS

    with tmp_csv.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)

        frame_idx = 0
        while True:
            ok, frame_bgr = cap.read()
            if not ok or frame_bgr is None:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_rgb = np.ascontiguousarray(frame_rgb, dtype=np.uint8)

            # --- FIX TIMESTAMP ---
            timestamp_ms = int(round(frame_idx * 1000.0 / fps))
            if timestamp_ms <= last_ts:
                print("BUMP", video_path.name, frame_idx, timestamp_ms, last_ts)
                timestamp_ms = last_ts + 1
            last_ts = timestamp_ms
            # ----------------------

            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            if frame_idx < 5:
                print("DBG", video_path.name, frame_idx, timestamp_ms, last_ts)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            # defaults (IMPORTANT: never skip writing)
            l_vec, r_vec = nan63.copy(), nan63.copy()
            l_ok, r_ok = 0, 0
            l_sc, r_sc = 0.0, 0.0

            if result.hand_landmarks:
                for i, hand_lms in enumerate(result.hand_landmarks):
                    label = result.handedness[i][0].category_name
                    score = float(result.handedness[i][0].score)
                    vec = to63(hand_lms)

                    if label == "Left":
                        l_vec, l_ok, l_sc = vec, 1, score
                    elif label == "Right":
                        r_vec, r_ok, r_sc = vec, 1, score

            writer.writerow([
                frame_idx, timestamp_ms,
                l_ok, r_ok,
                l_sc, r_sc,
                *l_vec.tolist(),
                *r_vec.tolist()
            ])

            rows_written += 1
            frame_idx += 1

    cap.release()

    if rows_written == 0:
        tmp_csv.unlink(missing_ok=True)
        return {
            "video": video_path.name,
            "status": "no_rows",
            "rows": 0,
            "total_frames": total_frames
        }

    tmp_csv.replace(out_csv)

    return {
        "video": video_path.name,
        "status": "ok",
        "rows": rows_written,
        "fps": fps,
        "total_frames": total_frames,
        "out_csv": str(out_csv)
    }

# ---------- batch run ----------

summaries = []

for vp in tqdm(video_files, desc="Generating per-video CSVs"):
    with HandLandmarker.create_from_options(options) as landmarker:
        out_csv = OUT_DIR / f"{vp.stem}.csv"

        if out_csv.exists() and csv_has_data_rows(out_csv):
            summaries.append({
                "video": vp.name,
                "status": "skipped_has_rows",
                "rows": None
            })
            continue

        summary = process_video_to_csv_working(vp, out_csv, landmarker)
        summaries.append(summary)

df = pd.DataFrame(summaries)
df

Generating per-video CSVs:   0%|                         | 0/20 [00:00<?, ?it/s]I0000 00:00:1769845281.237055 3589135 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1769845281.244374 3589137 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769845281.260738 3589139 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


DBG 00335.mp4 0 0 0
DBG 00335.mp4 1 40 40
DBG 00335.mp4 2 80 80
DBG 00335.mp4 3 120 120
DBG 00335.mp4 4 160 160


Generating per-video CSVs:   5%|▊                | 1/20 [00:01<00:19,  1.01s/it]I0000 00:00:1769845282.247131 3589172 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1769845282.250713 3589174 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769845282.255687 3589174 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


DBG 00336.mp4 0 0 0
DBG 00336.mp4 1 33 33
DBG 00336.mp4 2 67 67
DBG 00336.mp4 3 100 100
DBG 00336.mp4 4 133 133


Generating per-video CSVs:  10%|█▋               | 2/20 [00:02<00:23,  1.31s/it]I0000 00:00:1769845283.761195 3589214 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1769845283.763872 3589216 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769845283.767049 3589216 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

DBG 00435.mp4 0 0 0
DBG 00435.mp4 1 33 33
DBG 00435.mp4 2 66 66
DBG 00435.mp4 3 99 99
DBG 00435.mp4 4 132 132


Generating per-video CSVs:  95%|███████████████▏| 19/20 [00:28<00:01,  1.76s/it]I0000 00:00:1769845309.353240 3590083 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1769845309.356097 3590087 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769845309.359522 3590087 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


DBG 00583.mp4 0 0 0
DBG 00583.mp4 1 33 33
DBG 00583.mp4 2 67 67
DBG 00583.mp4 3 100 100
DBG 00583.mp4 4 133 133


Generating per-video CSVs: 100%|████████████████| 20/20 [00:29<00:00,  1.46s/it]


,video,status,rows,fps,total_frames,out_csv
0,00335.mp4,ok,58,25.000000,58,output/00335.csv
1,00336.mp4,ok,65,30.004616,65,output/00336.csv
2,00338.mp4,ok,71,29.970000,72,output/00338.csv
3,00339.mp4,ok,60,29.970000,61,output/00339.csv
4,00341.mp4,ok,84,30.331450,84,output/00341.csv
5,00376.mp4,ok,56,25.000000,56,output/00376.csv
6,00377.mp4,ok,89,30.003371,89,output/00377.csv
7,00381.mp4,ok,36,29.970000,37,output/00381.csv
8,00382.mp4,ok,44,29.970000,46,output/00382.csv
9,00384.mp4,ok,85,30.327147,85,output/00384.csv


# 2nd go

In [6]:
from pathlib import Path
import csv
import cv2
import numpy as np
from tqdm import tqdm
import pandas as pd
import mediapipe as mp

VIDEOS_DIR = Path("videos")
OUT_DIR = Path("output_holistic")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_VIDEOS = 20
video_files = sorted(VIDEOS_DIR.glob("*.mp4"))[:MAX_VIDEOS]

# MediaPipe holistic
mp_holistic = mp.solutions.holistic

In [10]:
def csv_has_data_rows(csv_path: Path) -> bool:
    try:
        with csv_path.open("r", encoding="utf-8") as f:
            if next(f, None) is None:  # header
                return False
            return next(f, None) is not None  # at least one row
    except Exception:
        return False


def pose_to132(pose_landmarks):
    """33 pose landmarks -> 132 (x,y,z,visibility)*33"""
    if pose_landmarks is None:
        return np.full((33*4,), np.nan, dtype=np.float32), 0
    arr = []
    for lm in pose_landmarks.landmark:
        arr.extend([lm.x, lm.y, lm.z, lm.visibility])
    return np.array(arr, dtype=np.float32), 1


def hand_to63(hand_landmarks):
    """21 hand landmarks -> 63 (x,y,z)*21"""
    if hand_landmarks is None:
        return np.full((21*3,), np.nan, dtype=np.float32), 0
    arr = []
    for lm in hand_landmarks.landmark:
        arr.extend([lm.x, lm.y, lm.z])
    return np.array(arr, dtype=np.float32), 1

In [12]:
def process_video_to_csv_holistic(video_path: Path, out_csv: Path, holistic) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"video": video_path.name, "status": "open_failed", "rows": 0, "fps": None, "total_frames": None, "out_csv": str(out_csv)}

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    header = (
        ["frame", "timestamp_ms",
         "pose_present", "left_hand_present", "right_hand_present",
         "pose_score", "left_hand_score", "right_hand_score"] +
        [f"pose_lm{i}_{a}" for i in range(33) for a in ("x","y","z","vis")] +
        [f"left_lm{i}_{a}" for i in range(21) for a in ("x","y","z")] +
        [f"right_lm{i}_{a}" for i in range(21) for a in ("x","y","z")]
    )

    rows_written = 0
    bad_frames = 0
    frame_idx = 0

    with out_csv.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)

        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Fix A style timestamp: monotonic, integer ms
            timestamp_ms = frame_idx  # (we don't actually need true ms for solutions)

            # Run holistic
            res = holistic.process(frame_rgb)

            pose_vec, pose_ok = pose_to132(res.pose_landmarks)
            left_vec, left_ok = hand_to63(res.left_hand_landmarks)
            right_vec, right_ok = hand_to63(res.right_hand_landmarks)

            # Scores: Solutions API doesn't give handedness confidence like Tasks.
            # We'll use a simple heuristic score: 1.0 if present else 0.0
            pose_score = 1.0 if pose_ok else 0.0
            left_score = 1.0 if left_ok else 0.0
            right_score = 1.0 if right_ok else 0.0

            try:
                w.writerow([
                    int(frame_idx), int(timestamp_ms),
                    int(pose_ok), int(left_ok), int(right_ok),
                    float(pose_score), float(left_score), float(right_score),
                    *pose_vec.tolist(),
                    *left_vec.tolist(),
                    *right_vec.tolist(),
                ])
                rows_written += 1
            except Exception:
                bad_frames += 1

            frame_idx += 1

    cap.release()

    return {
        "video": video_path.name,
        "status": "ok",
        "rows": rows_written,
        "bad_frames": bad_frames,
        "fps": float(fps),
        "total_frames": total_frames,
        "out_csv": str(out_csv),
    }

In [14]:
summaries = []

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    refine_face_landmarks=False,  # keep it light; turn on later if needed
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as holistic:

    for vp in tqdm(video_files, desc="Generating per-video Holistic CSVs"):
        out_csv = OUT_DIR / f"{vp.stem}.csv"

        if out_csv.exists() and csv_has_data_rows(out_csv):
            summaries.append({"video": vp.name, "status": "skipped_has_rows", "rows": None, "out_csv": str(out_csv)})
            continue

        summary = process_video_to_csv_holistic(vp, out_csv, holistic)
        summaries.append(summary)

df = pd.DataFrame(summaries)
df

I0000 00:00:1769847891.187833 3636689 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Generating per-video Holistic CSVs:   0%|                | 0/20 [00:00<?, ?it/s]W0000 00:00:1769847891.248493 3640248 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769847891.255303 3640253 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769847891.256337 3640250 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769847891.256352 3640252 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:17698478

,video,status,rows,bad_frames,fps,total_frames,out_csv
0,00335.mp4,ok,58,0,25.000000,58,output_holistic/00335.csv
1,00336.mp4,ok,65,0,30.004616,65,output_holistic/00336.csv
2,00338.mp4,ok,71,0,29.970000,72,output_holistic/00338.csv
3,00339.mp4,ok,60,0,29.970000,61,output_holistic/00339.csv
4,00341.mp4,ok,84,0,30.331450,84,output_holistic/00341.csv
5,00376.mp4,ok,56,0,25.000000,56,output_holistic/00376.csv
6,00377.mp4,ok,89,0,30.003371,89,output_holistic/00377.csv
7,00381.mp4,ok,36,0,29.970000,37,output_holistic/00381.csv
8,00382.mp4,ok,44,0,29.970000,46,output_holistic/00382.csv
9,00384.mp4,ok,85,0,30.327147,85,output_holistic/00384.csv
